In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':20})
import os
import pickle
import torch
import json

In [ ]:
data_dir = '../Data'
# asv_name = 'tblcounts_asv_wide.csv'
# asv_path = os.path.join(data_dir, asv_name)
# asv_df = pd.read_csv(asv_path)

In [3]:
drugs_name = 'tbldrug.csv'
drugs_path = os.path.join(data_dir, drugs_name)
drugs_df = pd.read_csv(drugs_path)

/tmp/ipykernel_16981/2872359644.py:3: DtypeWarning: Columns (0: PatientID) have mixed types. Specify dtype option on import or set low_memory=False.
  drugs_df = pd.read_csv(drugs_path)


In [4]:
samples_name = 'tblASVsamples.csv'
samples_path = os.path.join(data_dir, samples_name)
samples_df = pd.read_csv(samples_path)

In [ ]:
# print(asv_df.head())
print(drugs_df.head())
print(samples_df.head())

  PatientID  StartTimepoint  StopTimepoint         Factor  \
0      1000            -160           -160  ciprofloxacin   
1      1000            -160           -160    fluconazole   
2      1000            -151           -151      aztreonam   
3      1000            -151           -151     vancomycin   
4      1000            -150           -150      aztreonam   

                    Category        Route  StartDayRelativeToNearestHCT  \
0                 quinolones  intravenous                          -169   
1                antifungals  intravenous                          -169   
2  miscellaneous antibiotics  intravenous                          -160   
3   glycopeptide antibiotics  intravenous                          -160   
4  miscellaneous antibiotics  intravenous                          -159   

   StopDayRelativeToNearestHCT  
0                         -169  
1                         -169  
2                         -160  
3                         -160  
4                

In [9]:
# 1. Load the datasets
samples_df['PatientID'] = samples_df['PatientID'].astype(str)
drugs_df['PatientID'] = drugs_df['PatientID'].astype(str)

# 2. Sort chronologically
samples_df = samples_df.sort_values(by=['PatientID', 'DayRelativeToNearestHCT'])
drugs_df = drugs_df.sort_values(by=['PatientID', 'StartDayRelativeToNearestHCT'])

trajectory_data = []

# 3. Iterate over every sample, treating each as a potential Target (Sample Y)
for _, row_y in samples_df.iterrows():
    patient_id = row_y['PatientID']
    sample_y = row_y['SampleID']
    day_y = row_y['DayRelativeToNearestHCT']
    
    # Define the 30-day lookback window: [day_y - 30, day_y)
    window_start = day_y - 30
    
    # 4. Isolate this patient's data
    patient_samples = samples_df[samples_df['PatientID'] == patient_id]
    patient_drugs = drugs_df[drugs_df['PatientID'] == patient_id]
    
    # 5. Extract all prior samples within the 30-day window
    prior_samples = patient_samples[
        (patient_samples['DayRelativeToNearestHCT'] >= window_start) & 
        (patient_samples['DayRelativeToNearestHCT'] < day_y)
    ]
    
    # 6. Extract all drugs administered within the 30-day window
    # We include any drug that started within the window
    prior_drugs = patient_drugs[
        (patient_drugs['StartDayRelativeToNearestHCT'] >= window_start) & 
        (patient_drugs['StartDayRelativeToNearestHCT'] < day_y)
    ]
    
    # Format the histories as lists of dictionaries for easy parsing later
    history_samples = prior_samples[['SampleID', 'DayRelativeToNearestHCT']].to_dict('records')
    history_drugs = prior_drugs[['Factor', 'StartDayRelativeToNearestHCT', 'StopDayRelativeToNearestHCT']].to_dict('records')
    
    trajectory_data.append({
        'PatientID': patient_id,
        'SampleID_Y': sample_y,
        'Day_Y': day_y,
        'Num_Prior_Samples': len(prior_samples),
        'Num_Prior_Drugs': len(prior_drugs),
        # Convert lists to JSON strings so they store cleanly in a pandas DataFrame/CSV
        'Prior_Samples_30d': json.dumps(history_samples),
        'Prior_Drugs_30d': json.dumps(history_drugs)
    })

# 7. Create the final DataFrame
trajectory_df = pd.DataFrame(trajectory_data)

# Optional: Filter out samples that have zero prior history (no samples AND no drugs)
# trajectory_df = trajectory_df[(trajectory_df['Num_Prior_Samples'] > 0) | (trajectory_df['Num_Prior_Drugs'] > 0)]

print(trajectory_df.head())

  PatientID SampleID_Y  Day_Y  Num_Prior_Samples  Num_Prior_Drugs  \
0      1000      1000A   -9.0                  0                4   
1      1000      1000B   -4.0                  1                7   
2      1000      1000C    6.0                  2                8   
3      1000      1000D    9.0                  3               10   
4      1000      1000E   13.0                  4               11   

                                   Prior_Samples_30d  \
0                                                 []   
1  [{"SampleID": "1000A", "DayRelativeToNearestHC...   
2  [{"SampleID": "1000A", "DayRelativeToNearestHC...   
3  [{"SampleID": "1000A", "DayRelativeToNearestHC...   
4  [{"SampleID": "1000A", "DayRelativeToNearestHC...   

                                     Prior_Drugs_30d  
0  [{"Factor": "sulfamethoxazole trimethoprim", "...  
1  [{"Factor": "sulfamethoxazole trimethoprim", "...  
2  [{"Factor": "sulfamethoxazole trimethoprim", "...  
3  [{"Factor": "sulfamethoxa

In [11]:
trajectory_df.to_csv(os.path.join(data_dir, 'trajectory_data.csv'), index=False)

In [10]:
len(trajectory_df)

12732

In [13]:
print(trajectory_df[trajectory_df['PatientID']=='986'])

     PatientID SampleID_Y  Day_Y  Num_Prior_Samples  Num_Prior_Drugs  \
9023       986       986A    3.0                  0                7   
9024       986       986B   17.0                  1               15   

                                      Prior_Samples_30d  \
9023                                                 []   
9024  [{"SampleID": "986A", "DayRelativeToNearestHCT...   

                                        Prior_Drugs_30d  
9023  [{"Factor": "sulfamethoxazole trimethoprim", "...  
9024  [{"Factor": "sulfamethoxazole trimethoprim", "...  
